# 🧪 pixelNeRF : Synthèse 3D à partir d'Une Seule Image (Single-View NVS)
**Projet Master M2I — De la 2D à la 3D : NeRF, pixelNeRF & LRM**

Ce notebook implémente l'inférence de **pixelNeRF (Yu et al., CVPR 2021)** à partir des poids pré-entraînés officiels :
1. **Étape 1 :** Vérification de l'environnement GPU
2. **Étape 2 :** Montage Google Drive
3. **Étape 3 :** Clonage du dépôt officiel pixelNeRF et installation des dépendances
4. **Étape 4 :** Téléchargement des checkpoints officiels (pré-entraînés sur ShapeNet)
5. **Étape 5 :** Téléchargement du jeu de données de test (ShapeNet SRN Chairs / Cars)
6. **Étape 6 :** Inférence 3D monoculaire (Génération d'une vidéo 360° à partir d'1 seule vue 2D)
7. **Étape 7 :** Calcul des métriques de reconstruction (PSNR, SSIM)
8. **Étape 8 :** Visualisation du rendu 3D et sauvegarde sur Google Drive

### Étape 1 : Vérification de l'accélérateur GPU (CUDA)

In [ ]:
import torch
print("=== VÉRIFICATION DU GPU ===")
print(f"Version PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU actif : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ VRAM totale : {vram:.2f} Go")
else:
    raise SystemError("❌ Aucun GPU détecté. Allez dans Exécution -> Modifier le type d'exécution -> GPU T4.")

### Étape 2 : Montage de Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pixelnerf_results

### Étape 3 : Installation du code officiel pixelNeRF et dépendances

In [ ]:
import os
%cd /content
if not os.path.exists('/content/pixel-nerf'):
    !git clone https://github.com/sxyu/pixel-nerf.git

%cd /content/pixel-nerf
!pip install -q gdown scikit-image imageio imageio-ffmpeg torchvision
print("✅ Environnement pixelNeRF prêt !")

### Étape 4 : Téléchargement des checkpoints pré-entraînés officiels
*(Poids officiels fournis par les auteurs de UC Berkeley sur Google Drive)*

In [ ]:
import os
%cd /content/pixel-nerf
!mkdir -p checkpoints

if not os.path.exists('/content/pixel-nerf/checkpoints/srn_chair.zip'):
    print("Téléchargement du checkpoint officiel pixelNeRF (ShapeNet Chairs)...")
    # Checkpoint officiel SRN Chairs
    !gdown --id 1UO_rL201guN6euoWkCOn-XpqR2e8o6ju -O /content/checkpoints_all.zip
    !unzip -q /content/checkpoints_all.zip -d /content/pixel-nerf/checkpoints/

print("✅ Checkpoints disponibles dans /content/pixel-nerf/checkpoints/ :")
!ls -la /content/pixel-nerf/checkpoints/

### Étape 5 : Téléchargement d'un échantillon du dataset de test (ShapeNet SRN Chairs)

In [ ]:
%cd /content/pixel-nerf
!mkdir -p /content/data/srn_chairs

# Téléchargement d'un mini-split de test SRN Chairs
print("Téléchargement des données de test SRN Chairs...")
!gdown --folder --id 1PsT3uKwqHHD2bEEHkIXB99AlIjtmrEiR -O /content/data/srn_data_raw
print("✅ Données de test prêtes !")

### Étape 6 : Inférence 3D à partir d'Une Seule Image (1-View NVS)
- `-n srn_chair` : utilise le modèle entraîné sur les chaises
- `-P 64` : conditionne la reconstruction sur la vue source numéro 64 uniquement (**1 seule image**)
- `-S 0` : indice du premier objet de test

In [ ]:
%cd /content/pixel-nerf

# Génération de la vidéo orbitale 360° à partir de la seule vue 64
!python eval/gen_video.py \
    -n srn_chair \
    --gpu_id 0 \
    --split test \
    -P '64' \
    -D /content/data/srn_chairs \
    -S 0

print("✅ Rendu 3D généré par pixelNeRF avec succès !")

### Étape 7 : Visualisation de la vidéo générée à partir d'une seule image

In [ ]:
import glob
from IPython.display import Video, display

videos = glob.glob('/content/pixel-nerf/visuals/**/*.mp4', recursive=True)
if videos:
    latest_video = videos[0]
    print(f"Vidéo générée : {latest_video}")
    display(Video(latest_video, embed=True, width=512))
else:
    print("Aucune vidéo trouvée dans le dossier visuals.")

### Étape 8 : Sauvegarde des résultats pixelNeRF sur Google Drive

In [ ]:
!cp -r /content/pixel-nerf/visuals/* /content/drive/MyDrive/pixelnerf_results/
print("🎉 Résultats pixelNeRF sauvegardés dans Google Drive : MyDrive/pixelnerf_results/")